# Audio Segmentation & Static Masking Pipeline

This Colab provides an end-to-end operational pipeline for clustering VHF/UHF radio communication segments, extracting active speech blocks, and selectively masking untranscribed non-speech acoustic events with normalized pink noise to construct highly accurate target datasets for Supervised Fine-Tuning (SFT) and model performance evaluation.

---

## 🏛️ Technical Pipeline Architecture & Operational Taxonomy

### 1. Flexible Prefix Annotation Matching (`str.startswith`)
* **The Parsing Strategy**: Human annotators frequently export composite sub-classification labels (e.g., `PII/ADDRESS`, `PII/ID/PHONE`, `PII/MEDICAL`). 
* **Implementation**: The pipeline replaces strict exact string equality checks with prefix matching (`str.startswith`). This ensures that composite or multi-level annotations are reliably captured across filtering and masking loops.

### 2. Domain-Adapted SFT Masking Taxonomy
To balance pristine target text alignment with real-world acoustic adaptation, the pipeline establishes a highly specialized audio masking profile:

* **Preserved Real-World Signaling**: Automated automated Quik-Call two-tone pager alerts (`RINGING`), dual-tone multi-frequency automated repeater tones (`DTMF`), and ambient RF carrier squelch/static remain **completely unmasked**. Preserving these features allows fine-tuned ASR models to learn authentic emergency dispatch pre-cursors and refrain from hallucinating speech when evaluating automated signaling.
* **Targeted Pink Noise Masking**: The pipeline specifically targets and replaces exactly five categories of untranscribed or sensitive audio with normalized `1/f` Gaussian pink noise:
  1. `PII*`: All composite variants of sensitive personal identifiers, names, phone numbers, and addresses.
  2. `UNINTELLIGIBLE`: Extremely noisy, obscured, or garbled human speech that cannot be reliably transcribed.
  3. `FOREIGN_SPEECH`: Extraneous non-English cross-talk or amateur radio bleed-over.
  4. `UNKNOWN`: Unclassifiable human voice transmissions.
  5. `LAUGHTER`: Extraneous untranscribed human vocalizations.

### 3. Acoustic Safeguards & Speech Padding
* **Expanded Transients**: Mask boundaries are automatically expanded by `50ms` in both directions to entirely swallow unannotated click transients or sudden acoustic pops.
* **Core Speech Protection**: Expanded pink noise masks are strictly clamped against known `TRANSCRIPTION` timestamps so they never overwrite or bleed into valid Ground Truth human speech.
* **Zero-Crossing Crossfades**: Applies 10ms linear crossfade ramps to the good audio immediately outside the masking boundary to seamlessly blend the synthetic pink noise into the RF timeline without introducing digital click artifacts.


In [ ]:
# @title Install dependencies
%pip install -q --upgrade \
    loguru \
    soundfile \
    colorednoise

In [ ]:
# @title Imports
from collections import defaultdict
import json
from pathlib import Path
import sys
from urllib.parse import urlparse

import colorednoise as cn
from google.cloud import storage
from google.colab import auth, userdata
from IPython.display import display
from loguru import logger
import numpy as np
import pandas as pd
import soundfile as sf
from tqdm.notebook import tqdm

In [ ]:
# @title Authentication and client initialization
auth.authenticate_user()


# User Configuration from Colab Secrets
GCP_PROJECT_ID = userdata.get("GCP_PROJECT_ID")
GCS_BUCKET = userdata.get("GCS_BUCKET")

!gcloud config set project {GCP_PROJECT_ID} --quiet

# Initialize GCS Client
gcs_client = storage.Client(project=GCP_PROJECT_ID)

In [ ]:
# @title Pipeline Configuration

# fmt: off
# @markdown ### 1. GCS Input & Output Paths
# @markdown Path to the manifest file relative to the 'manifests/' directory (e.g. fire_notifications/eval/manifest.json)
MANIFEST_FILE_PATH = ""  # @param {type:"string"}
# @markdown Path to the audio directory relative to the 'audio/' directory (e.g. fire_notifications/eval)
AUDIO_DIR_PATH = ""  # @param {type:"string"}
# @markdown Base GCS path for output relative to the 'segmented_audio/' directory (e.g. fire_notifications_masked or echo/eval_masked)
OUTPUT_PATH = ""  # @param {type:"string"}
# Overwrite existing GCS files?
OVERWRITE_EXISTING = False  # @param {type:"boolean"}

# @markdown ### 2. Segmentation Parameters
# Maximum gap allowed before splitting into a new transmission block (seconds)
TRANSMISSION_GAP_THRESHOLD = 1.0  # @param {type:"slider", min:0.1, max:3.0, step:0.05}
# Collision-Aware Padding (ie, max padding value)
DESIRED_PAD = 0.5  # @param {type:"slider", min:0.0, max:2.0, step:0.1}
# Export speechless/noise-only transmissions (GT "") for False-Positive Rejection Training
EXPORT_SPEECHLESS_TRANSMISSIONS = False  # @param {type:"boolean"}

# @markdown ### 3. System Settings
LOCAL_BASE_PATH = "/content"  # @param {type:"string"}
LOG_LEVEL = "WARNING"  # @param ["DEBUG", "INFO", "WARNING", "ERROR"]
# fmt: on

assert MANIFEST_FILE_PATH, (
    "MANIFEST_FILE_PATH must be provided and cannot be empty."
)
assert AUDIO_DIR_PATH, "AUDIO_DIR_PATH must be provided and cannot be empty."
assert OUTPUT_PATH, "OUTPUT_PATH must be provided and cannot be empty."

CACHE_DIR = f"{LOCAL_BASE_PATH}/raw_audio"
SEGMENTS_DIR = f"{LOCAL_BASE_PATH}/segmented_audio_masked"
BATCH_MANIFEST_FILENAME = "batch_manifest.jsonl"
BATCH_MANIFEST_ALL_FILENAME = "batch_manifest_all.jsonl"

# Initialize loguru
logger.remove()
logger.add(
    sys.stderr, level=LOG_LEVEL, format="<level>{level}</level>: {message}"
)

Path(CACHE_DIR).mkdir(parents=True, exist_ok=True)
Path(SEGMENTS_DIR).mkdir(parents=True, exist_ok=True)

In [ ]:
# @title Transmission Building & Static Masking Logic
def ensure_local_gcs_audio(gcs_uri: str) -> str:
    """Downloads raw audio to local cache and validates file integrity."""
    parsed = urlparse(gcs_uri)
    bucket_name = parsed.netloc
    blob_name = parsed.path.lstrip("/")
    filename = Path(blob_name).name
    local_path = Path(CACHE_DIR) / filename

    # If file exists but is 0 bytes, it's corrupt. Delete it.
    if local_path.exists() and local_path.stat().st_size == 0:
        logger.warning(
            f"Found empty file {filename}, removing for re-download..."
        )
        local_path.unlink()

    if not local_path.exists():
        logger.info(f"Downloading {filename} from GCS...")
        try:
            gcs_client.bucket(bucket_name).blob(blob_name).download_to_filename(
                str(local_path)
            )
        except Exception as e:
            logger.error(f"Failed to download {filename} from GCS: {e}")
            raise e
    return str(local_path)


def cleanup_gcs_output(bucket_name: str, prefix: str) -> None:
    """Deletes the GCS 'directory' (prefix) and all contents recursively."""
    bucket = gcs_client.bucket(bucket_name)
    blobs = list(bucket.list_blobs(prefix=prefix))

    if blobs:
        logger.info(
            f"Deleting prefix and all contents: gs://{bucket_name}/{prefix} ({len(blobs)} files)..."
        )
        bucket.delete_blobs(blobs)
        logger.info("Cleanup complete.")
    else:
        logger.info(
            f"Target prefix gs://{bucket_name}/{prefix} is already empty."
        )


def build_transmissions(
    anchor_segments: list[dict], threshold: float
) -> list[list[dict]]:
    """Clusters ONLY speech segments into dense transmissions of activity."""
    if not anchor_segments:
        return []

    transmissions = []
    current_transmission = [anchor_segments[0]]
    max_end = anchor_segments[0]["offset"] + anchor_segments[0]["duration"]

    for curr_seg in anchor_segments[1:]:
        gap = curr_seg["offset"] - max_end

        if gap <= threshold:
            current_transmission.append(curr_seg)
            max_end = max(max_end, curr_seg["offset"] + curr_seg["duration"])
        else:
            transmissions.append(current_transmission)
            current_transmission = [curr_seg]
            max_end = curr_seg["offset"] + curr_seg["duration"]

    transmissions.append(current_transmission)
    return transmissions

In [ ]:
# @title Execute Execute Segmentation & Masking Pipeline
def generate_pink_noise(samples: int, volume_scale: float = 0.1) -> np.ndarray:
    """Generates pink noise using the specialized colorednoise library."""
    if samples <= 0:
        return np.array([], dtype=np.float32)
    # beta=1 for pink noise (1/f)
    pink_noise = cn.powerlaw_psd_gaussian(1, samples)

    # 1. Remove DC offset (Center the waveform at zero)
    pink_noise = pink_noise - np.mean(pink_noise)

    # 2. Normalize and scale
    if np.max(np.abs(pink_noise)) > 0:
        pink_noise = (pink_noise / np.max(np.abs(pink_noise))) * volume_scale
    return pink_noise.astype(np.float32)


def subtract_sample_intervals(
    target: tuple[int, int], subtract_list: list[tuple[int, int]]
) -> list[tuple[int, int]]:
    """Subtracts a list of sample intervals from a target interval, ensuring absolute protection."""
    t_start, t_end = target
    if t_start >= t_end:
        return []

    # Sort subtract list and merge overlapping ones
    merged_sub = []
    for s, e in sorted(subtract_list):
        if not merged_sub or merged_sub[-1][1] < s:
            merged_sub.append([s, e])
        else:
            merged_sub[-1][1] = max(merged_sub[-1][1], e)

    results = []
    curr_start = t_start
    for s, e in merged_sub:
        if e <= curr_start:
            continue
        if s >= t_end:
            break
        if s > curr_start:
            results.append((curr_start, s))
        curr_start = max(curr_start, e)

    if curr_start < t_end:
        results.append((curr_start, t_end))
    return results


def run_segmentation_pipeline() -> None:
    full_output_prefix = f"segmented_audio/{OUTPUT_PATH.strip('/')}"

    # Using existing constants directly
    if OVERWRITE_EXISTING:
        cleanup_gcs_output(GCS_BUCKET, full_output_prefix)

    output_bucket = gcs_client.bucket(GCS_BUCKET)

    full_manifest_path = f"manifests/{MANIFEST_FILE_PATH.lstrip('/')}"
    m_blob = output_bucket.blob(full_manifest_path)
    content = m_blob.download_as_text()

    if not content or len(content.strip()) < 2:
        logger.error(f"Manifest at {full_manifest_path} appears to be empty.")
        return

    try:
        manifest_data = json.loads(content)
    except json.JSONDecodeError as e:
        logger.error(f"Failed to parse manifest: {e}")
        return

    files_to_process = defaultdict(list)
    for entry in manifest_data:
        files_to_process[entry["audio_filepath"]].append(entry)

    final_manifest_entries = []

    # 🌿 Target and Pink-Noise Mask ALL extraneous untranscribed human vocalizations and sensitive profiles
    mask_bases = (
        "PII",
        "UNINTELLIGIBLE",
        "FOREIGN_SPEECH",
        "UNKNOWN",
        "LAUGHTER",
    )

    for json_audio_path, raw_segments in tqdm(
        files_to_process.items(), desc="Processing Audio Files"
    ):
        filename = Path(json_audio_path).name
        example_id = Path(filename).stem
        # Use flexible audio path structure relative to the bucket and audio directory
        true_gcs_uri = (
            f"gs://{GCS_BUCKET}/audio/{AUDIO_DIR_PATH.strip('/')}/{filename}"
        )

        raw_segments.sort(key=lambda x: x["offset"])
        local_src_path = ensure_local_gcs_audio(true_gcs_uri)

        y, sr = sf.read(local_src_path)
        if y.ndim > 1:  # Ensure mono
            y = y.mean(axis=1)

        logger.info(f"Processing {example_id}...")

        # Beautifully cluster ALL audio activity segments (human vocalizations, DTMF, ringing, static) into harvested transmission blocks
        clustered_segments = build_transmissions(
            raw_segments, TRANSMISSION_GAP_THRESHOLD
        )

        # Stage 1: Smart Retention Criteria
        # To be retained, a transmission block MUST contain at least one golden speech or standalone mechanical signal (TRANSCRIPTION, DTMF, RINGING).
        # Blocks that consist entirely of STATIC or pink-noise masked profiles (PII, UNINTELLIGIBLE, etc.) are permanently discarded.
        valid_standalone_bases = ("TRANSCRIPTION", "DTMF", "RINGING")
        valid_clustered_segments = [
            block
            for block in clustered_segments
            if any(
                seg.get("category", "")
                .strip()
                .upper()
                .startswith(valid_standalone_bases)
                for seg in block
            )
        ]

        # Stage 2: Stray Rejection Toggle
        final_segments = [
            block
            for block in valid_clustered_segments
            if EXPORT_SPEECHLESS_TRANSMISSIONS
            or any(
                seg.get("category", "").strip().upper() == "TRANSCRIPTION"
                for seg in block
            )
        ]

        max_audio_time = len(y) / sr

        for i, segment_block in enumerate(final_segments):
            # 1. Extract Ground Truth text
            gt_texts = [
                cleaned
                for seg in segment_block
                if seg.get("category", "").strip().upper() == "TRANSCRIPTION"
                and (cleaned := seg.get("text", "").strip())
            ]
            segment_gt_text = " ".join(gt_texts).strip()

            # 2. Identify the core audio transmission boundaries (including leading/trailing non-transcription sounds in the block)
            transcription_segs = [
                s
                for s in segment_block
                if s.get("category", "").strip().upper() == "TRANSCRIPTION"
            ]
            core_start = min(s["offset"] for s in segment_block)
            core_end = max(s["offset"] + s["duration"] for s in segment_block)

            # Check for padding collisions against ALL labeled segments in the file (preventing bleed into stray DTMF, ringing, or other transmissions)
            preceding_segs = [
                s
                for s in raw_segments
                if s["offset"] + s["duration"] <= core_start
            ]
            available_front_gap = (
                core_start
                - max(s["offset"] + s["duration"] for s in preceding_segs)
                if preceding_segs
                else DESIRED_PAD
            )
            safe_front_pad = max(0.0, min(DESIRED_PAD, available_front_gap))

            following_segs = [
                s for s in raw_segments if s["offset"] >= core_end
            ]
            available_back_gap = (
                min(s["offset"] for s in following_segs) - core_end
                if following_segs
                else DESIRED_PAD
            )
            safe_back_pad = max(0.0, min(DESIRED_PAD, available_back_gap))

            # 3. Final Audio Bounds
            segment_start = max(0.0, core_start - safe_front_pad)
            segment_end = min(max_audio_time, core_end + safe_back_pad)

            segment_id = f"{i:03d}"

            out_filename = f"{SEGMENTS_DIR}/{example_id}__seg{segment_id}.flac"
            blob_name = (
                f"{full_output_prefix}/{example_id}/{Path(out_filename).name}"
            )
            output_blob = output_bucket.blob(blob_name)

            has_speech = any(
                seg.get("category", "").strip().upper() == "TRANSCRIPTION"
                for seg in segment_block
            )

            if not OVERWRITE_EXISTING and output_blob.exists():
                final_manifest_entries.append(
                    {
                        "audio_filepath": f"gs://{GCS_BUCKET}/{blob_name}",
                        "text": segment_gt_text,
                        "example_id": example_id,
                        "segment_id": segment_id,
                        "offset": segment_start,
                        "duration": segment_end - segment_start,
                        "has_speech": has_speech,
                    }
                )
                continue

            start_samp = int(segment_start * sr)
            end_samp = int(segment_end * sr)
            segment_audio = y[start_samp:end_samp].copy()

            # Map all golden TRANSCRIPTION sample ranges for absolute masking exclusion
            golden_speech_intervals = []
            for t_seg in transcription_segs:
                t_start = t_seg["offset"]
                t_end = t_start + t_seg["duration"]
                if t_end > segment_start and t_start < segment_end:
                    gs_s = int((t_start - segment_start) * sr)
                    gs_e = int((t_end - segment_start) * sr)
                    golden_speech_intervals.append(
                        (max(0, gs_s), min(len(segment_audio), gs_e))
                    )

            # 🌿 Laser-focused masking loop entirely swallowing sensitive and confusing untranscribed profiles
            for seg in raw_segments:
                cat = seg.get("category", "").strip().upper()
                if cat.startswith(mask_bases):
                    # Expand mask by 50ms to swallow unannotated transients/mic clicks
                    mask_start = seg["offset"] - 0.05
                    mask_end = seg["offset"] + seg["duration"] + 0.05

                    if mask_end > segment_start and mask_start < segment_end:
                        raw_m_start = int((mask_start - segment_start) * sr)
                        raw_m_end = int((mask_end - segment_start) * sr)
                        raw_m_start = max(0, raw_m_start)
                        raw_m_end = min(len(segment_audio), raw_m_end)

                        # Subtract golden speech intervals to guarantee 100% protection
                        safe_sub_masks = subtract_sample_intervals(
                            (raw_m_start, raw_m_end), golden_speech_intervals
                        )

                        for m_start, m_end in safe_sub_masks:
                            target_len = m_end - m_start
                            if target_len <= 0:
                                continue

                            # 1. Calculate RMS before we mute anything
                            original_clip = segment_audio[m_start:m_end].copy()
                            rms_volume = (
                                np.sqrt(np.mean(original_clip**2))
                                if len(original_clip) > 0
                                else 0.05
                            )
                            # Clamp the RMS so the massive pop doesn't make our pink noise deafeningly loud
                            rms_volume = min(0.1, max(rms_volume, 0.04))

                            # 2. Define true crossfade boundaries (10ms outside the mask)
                            fade_len = int(sr * 0.01)
                            f_start = max(0, m_start - fade_len)
                            f_end = min(len(segment_audio), m_end + fade_len)

                            actual_fade_in_len = m_start - f_start
                            actual_fade_out_len = f_end - m_end

                            # 3. HARD MUTE the entire masked section to kill the transient pop instantly
                            segment_audio[m_start:m_end] = 0.0

                            # 4. Fade OUT the good audio just BEFORE the mask hits
                            if actual_fade_in_len > 0:
                                fade_out_curve = np.linspace(
                                    1, 0, actual_fade_in_len
                                )
                                segment_audio[f_start:m_start] *= fade_out_curve

                            # 5. Fade IN the good audio just AFTER the mask ends
                            if actual_fade_out_len > 0:
                                fade_in_curve = np.linspace(
                                    0, 1, actual_fade_out_len
                                )
                                segment_audio[m_end:f_end] *= fade_in_curve

                            # 6. Generate pink noise for the ENTIRE region (fades + mask)
                            noise_len = f_end - f_start
                            noise = generate_pink_noise(
                                noise_len, volume_scale=rms_volume * 3.0
                            )

                            # 7. Apply crossfades to the noise so it seamlessly blends
                            if actual_fade_in_len > 0:
                                noise[:actual_fade_in_len] *= np.linspace(
                                    0, 1, actual_fade_in_len
                                )
                            if actual_fade_out_len > 0:
                                noise[-actual_fade_out_len:] *= np.linspace(
                                    1, 0, actual_fade_out_len
                                )

                            # 8. Mix the noise into the timeline
                            segment_audio[f_start:f_end] += noise

            sf.write(
                out_filename, segment_audio, sr, format="FLAC", subtype="PCM_16"
            )
            output_blob.upload_from_filename(out_filename)

            final_manifest_entries.append(
                {
                    "audio_filepath": f"gs://{GCS_BUCKET}/{blob_name}",
                    "text": segment_gt_text,
                    "example_id": example_id,
                    "segment_id": segment_id,
                    "offset": segment_start,
                    "duration": segment_end - segment_start,
                    "has_speech": has_speech,
                }
            )

    # 1. Export speech-only batch manifest (Canonical BATCH_MANIFEST_FILENAME for legacy plug-and-play)
    speech_only_entries = [
        e for e in final_manifest_entries if e.get("has_speech")
    ]
    local_manifest = Path(LOCAL_BASE_PATH) / BATCH_MANIFEST_FILENAME
    with open(local_manifest, "w") as f:
        for entry in speech_only_entries:
            f.write(json.dumps(entry) + "\n")

    output_bucket.blob(
        f"{full_output_prefix}/{BATCH_MANIFEST_FILENAME}"
    ).upload_from_filename(str(local_manifest))

    # 2. Export full dual-set batch manifest (BATCH_MANIFEST_ALL_FILENAME for False-Positive Rejection training)
    local_manifest_all = Path(LOCAL_BASE_PATH) / BATCH_MANIFEST_ALL_FILENAME
    with open(local_manifest_all, "w") as f:
        for entry in final_manifest_entries:
            f.write(json.dumps(entry) + "\n")

    output_bucket.blob(
        f"{full_output_prefix}/{BATCH_MANIFEST_ALL_FILENAME}"
    ).upload_from_filename(str(local_manifest_all))

    logger.info(
        f"Pipeline Complete. Exported {len(speech_only_entries)} speech segments and {len(final_manifest_entries) - len(speech_only_entries)} speechless segments."
    )

    if final_manifest_entries:
        df = pd.DataFrame(final_manifest_entries)
        display(
            df[["segment_id", "offset", "duration", "has_speech", "text"]].head(
                10
            )
        )

In [ ]:
# @title Create the segments and manifest file
run_segmentation_pipeline()